# Advanced Problems with Solutions: Custom Classes, Equality, and Hashing

This notebook is a problem-driven deep dive into Python's object equality and hashing model. It builds on the core rule:

> If `a == b`, then `hash(a) == hash(b)` must also be true.

The converse is not required: unequal objects may share a hash. Correctness depends on the equality/hash contract; performance depends on distributing unequal keys reasonably well.

## What you will practice

- Designing immutable, hashable value objects.
- Repairing classes that define equality incorrectly.
- Understanding `NotImplemented`, symmetry, subclasses, and heterogeneous comparisons.
- Avoiding mutable dictionary keys.
- Using `dataclass` options intentionally.
- Building canonical composite keys from nested data.
- Measuring collision costs.
- Separating process-local `hash()` values from stable external identifiers.
- Testing the hash contract with reusable assertions and randomized cases.
- Building production-style cache keys and partition keys.

All solutions use only Python's standard library.


## Setup and reusable testing helpers

A good test suite should verify equality laws and the mandatory implication from equality to equal hashes. It should also check dictionary and set behavior, because that is where broken designs become operational bugs.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from functools import cached_property, lru_cache
from collections.abc import Hashable, Mapping, Sequence, Set
from typing import Any, Iterable
from timeit import timeit
import hashlib
import json
import math
import os
import random
import subprocess
import sys


In [2]:
def assert_hash_contract(a: Any, b: Any) -> None:
    """Assert the mandatory equality/hash implication for two objects."""
    if a == b:
        assert hash(a) == hash(b), (
            f"Hash contract violated: {a!r} == {b!r}, "
            f"but hash values differ: {hash(a)} != {hash(b)}"
        )


def assert_equivalence_laws(a: Any, b: Any, c: Any) -> None:
    """Check common equality laws for a proposed value-object design."""
    assert a == a, "Equality must be reflexive"
    assert (a == b) == (b == a), "Equality should be symmetric"
    if a == b and b == c:
        assert a == c, "Equality must be transitive"
    assert_hash_contract(a, b)
    assert_hash_contract(b, c)
    assert_hash_contract(a, c)


def assert_mapping_round_trip(key: Hashable, equivalent_key: Hashable) -> None:
    """Equivalent keys should retrieve and overwrite the same dictionary entry."""
    d = {key: "first"}
    assert d[equivalent_key] == "first"
    d[equivalent_key] = "updated"
    assert len(d) == 1
    assert d[key] == "updated"

print("Python:", sys.version.split()[0])
print("Hash width:", sys.hash_info.width)
print("Hash modulus:", sys.hash_info.modulus)


Python: 3.13.7
Hash width: 64
Hash modulus: 2305843009213693951


---
## Problem 1 — Diagnose a class that became unhashable

The following class compares users by `user_id`. The author expects it to be usable as a dictionary key, but `hash(User(...))` raises `TypeError`.

### Tasks

1. Explain why defining `__eq__` changed hashability.
2. Implement `__hash__` correctly.
3. Return `NotImplemented` for unsupported comparison types instead of returning `False` immediately.
4. Verify dictionary retrieval with an equivalent instance.


In [3]:
class BrokenUser:
    def __init__(self, user_id: int, display_name: str):
        self.user_id = user_id
        self.display_name = display_name

    def __eq__(self, other: object) -> bool:
        if isinstance(other, BrokenUser):
            return self.user_id == other.user_id
        return False

print("BrokenUser.__hash__:", BrokenUser.__hash__)
try:
    print(hash(BrokenUser(10, "Ada")))
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


BrokenUser.__hash__: None
TypeError: unhashable type: 'BrokenUser'


### Solution

When a class defines `__eq__` but does not define `__hash__`, Python sets `__hash__ = None`. This prevents the inherited identity-based hash from violating the contract for distinct instances that now compare equal.

Only fields that participate in equality should participate in hashing. Here, `display_name` is intentionally excluded from both.


In [4]:
class User:
    def __init__(self, user_id: int, display_name: str):
        self.user_id = user_id
        self.display_name = display_name

    def __repr__(self) -> str:
        return f"User(user_id={self.user_id!r}, display_name={self.display_name!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, User):
            return NotImplemented
        return self.user_id == other.user_id

    def __hash__(self) -> int:
        return hash(self.user_id)

u1 = User(10, "Ada")
u2 = User(10, "A. Lovelace")
u3 = User(11, "Grace")

assert u1 == u2
assert u1 != u3
assert_hash_contract(u1, u2)
assert_mapping_round_trip(u1, u2)
print({u1: "profile"}[u2])


profile


---
## Problem 2 — Build a normalized, immutable email key

Email addresses are often compared case-insensitively in application-level identity systems. Design an `EmailKey` that:

- strips surrounding whitespace;
- lowercases the value for identity purposes;
- is immutable and hashable;
- rejects malformed input with a clear exception;
- preserves only the canonical representation.

Use a frozen dataclass and validate in `__post_init__`.


### Solution

A frozen dataclass is a strong default for value objects used as keys. During `__post_init__`, `object.__setattr__` may be used to store a normalized value exactly once.


In [5]:
@dataclass(frozen=True, slots=True)
class EmailKey:
    value: str

    def __post_init__(self) -> None:
        normalized = self.value.strip().casefold()
        if normalized.count("@") != 1:
            raise ValueError(f"Invalid email address: {self.value!r}")
        local, domain = normalized.split("@")
        if not local or not domain or "." not in domain:
            raise ValueError(f"Invalid email address: {self.value!r}")
        object.__setattr__(self, "value", normalized)

    def __str__(self) -> str:
        return self.value

k1 = EmailKey("  Ada@Example.COM ")
k2 = EmailKey("ada@example.com")

assert k1 == k2
assert hash(k1) == hash(k2)
assert_mapping_round_trip(k1, k2)
print(k1, hash(k1))


ada@example.com 5002583119214345351


---
## Problem 3 — Reproduce and repair mutable-key corruption

A mutable `Point` uses `(x, y)` for equality and hashing. It is inserted into a dictionary, then its `x` coordinate changes.

### Tasks

1. Demonstrate that iteration can still show the key even when lookup fails.
2. Explain why the entry is effectively stranded.
3. Replace the design with an immutable implementation.


In [6]:
class MutablePoint:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"MutablePoint({self.x}, {self.y})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, MutablePoint):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return hash((self.x, self.y))

p = MutablePoint(2, 3)
lookup = {p: "occupied"}
old_hash = hash(p)
p.x = 99
new_hash = hash(p)

print("old hash:", old_hash)
print("new hash:", new_hash)
print("dictionary contents:", list(lookup.items()))
print("p in lookup:", p in lookup)
print("equivalent current point in lookup:", MutablePoint(99, 3) in lookup)


old hash: 8409376899596376432
new hash: -3486579784104132882
dictionary contents: [(MutablePoint(99, 3), 'occupied')]
p in lookup: False
equivalent current point in lookup: False


### Solution

The dictionary stores the key according to the hash it had at insertion time. Mutation changes the hash used for future searches, so lookup begins from a different probe sequence. Iteration does not perform a key lookup; it walks occupied slots directly, which is why the stranded key remains visible.


In [7]:
@dataclass(frozen=True, slots=True)
class Point:
    x: int
    y: int

safe_point = Point(2, 3)
safe_lookup = {safe_point: "occupied"}
assert safe_lookup[Point(2, 3)] == "occupied"

try:
    safe_point.x = 99
except (AttributeError, TypeError) as exc:
    print(type(exc).__name__ + ":", exc)


FrozenInstanceError: cannot assign to field 'x'


---
## Problem 4 — Choose dataclass fields that define identity

A package artifact has immutable identity `(name, version, platform)`, but also carries mutable operational metadata such as download count and mirror URL.

Design a dataclass that:

- is safely hashable;
- compares artifacts only by identity fields;
- does not let metadata affect equality or hashing;
- allows metadata to change without invalidating dictionary membership.


### Solution

Use a frozen outer value object only when all fields must be immutable. Here, operational metadata must change, so a better design separates immutable identity from mutable state.


In [8]:
@dataclass(frozen=True, slots=True)
class ArtifactId:
    name: str
    version: str
    platform: str


@dataclass(slots=True)
class ArtifactRecord:
    identity: ArtifactId
    mirror_url: str
    download_count: int = 0

artifact_id = ArtifactId("numpy", "2.1.0", "linux-x86_64")
record = ArtifactRecord(artifact_id, "https://mirror-a.example")
registry = {artifact_id: record}

record.download_count += 1
record.mirror_url = "https://mirror-b.example"

assert registry[ArtifactId("numpy", "2.1.0", "linux-x86_64")].download_count == 1
print(registry[artifact_id])


ArtifactRecord(identity=ArtifactId(name='numpy', version='2.1.0', platform='linux-x86_64'), mirror_url='https://mirror-b.example', download_count=1)


---
## Problem 5 — Strict type equality versus subclass equality

Suppose `Money(10, "USD")` and `DiscountedMoney(10, "USD", 0.20)` inherit from one another. If the base class uses `isinstance`, a base instance may compare equal to a subclass instance even though the subclass carries extra identity-relevant data.

### Tasks

1. Show the risk.
2. Implement strict-type equality for the base value object.
3. Give the subclass its own complete identity.


In [9]:
class RiskyMoney:
    def __init__(self, amount: int, currency: str):
        self.amount = amount
        self.currency = currency

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, RiskyMoney):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __hash__(self) -> int:
        return hash((self.amount, self.currency))

class RiskyDiscountedMoney(RiskyMoney):
    def __init__(self, amount: int, currency: str, discount: float):
        super().__init__(amount, currency)
        self.discount = discount

base = RiskyMoney(10, "USD")
sub = RiskyDiscountedMoney(10, "USD", 0.20)
print("Risky equality:", base == sub, sub == base)


Risky equality: True True


### Solution

For closed value-object types, `type(self) is type(other)` is often safer than `isinstance`. It prevents a subclass with additional identity semantics from being silently collapsed into the base class's equivalence relation.


In [10]:
@dataclass(frozen=True, slots=True)
class Money:
    amount: int
    currency: str

    def __eq__(self, other: object) -> bool:
        if type(self) is not type(other):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __hash__(self) -> int:
        return hash((type(self), self.amount, self.currency))


@dataclass(frozen=True, slots=True, eq=False)
class DiscountedMoney(Money):
    discount: float

    def __eq__(self, other: object) -> bool:
        if type(self) is not type(other):
            return NotImplemented
        return (
            self.amount,
            self.currency,
            self.discount,
        ) == (
            other.amount,
            other.currency,
            other.discount,
        )

    def __hash__(self) -> int:
        return hash((type(self), self.amount, self.currency, self.discount))

m = Money(10, "USD")
d = DiscountedMoney(10, "USD", 0.20)
assert m != d
assert len({m, d}) == 2
print({m, d})


{Money(amount=10, currency='USD'), DiscountedMoney(amount=10, currency='USD', discount=0.2)}


---
## Problem 6 — Use `NotImplemented` correctly in heterogeneous comparisons

Create a `Vector2D` that can compare to another `Vector2D`. Do **not** make it equal to a raw tuple, even though doing so may seem convenient.

Then provide an explicit adapter for tuple lookup.

Why is this preferable to broad cross-type equality?


### Solution

Returning `NotImplemented` allows Python to try the reflected comparison and keeps unsupported types outside the equivalence relation. Explicit conversion is clearer and avoids surprising transitivity or symmetry failures across unrelated types.


In [11]:
@dataclass(frozen=True, slots=True)
class Vector2D:
    x: float
    y: float

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Vector2D):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return hash((self.x, self.y))

    @classmethod
    def from_pair(cls, pair: Sequence[float]) -> "Vector2D":
        if len(pair) != 2:
            raise ValueError("A 2D vector requires exactly two components")
        return cls(float(pair[0]), float(pair[1]))

vectors = {Vector2D(1.0, 2.0): "north-east"}
assert vectors[Vector2D.from_pair((1, 2))] == "north-east"
assert Vector2D(1, 2) != (1, 2)
print(vectors[Vector2D.from_pair([1, 2])])


north-east


---
## Problem 7 — Canonicalize nested mutable data into a hashable key

A caching layer receives configurations containing nested dictionaries, lists, sets, tuples, and primitive values. Build a `deep_freeze` function that converts equivalent structures into a deterministic, hashable representation.

Requirements:

- mapping order must not matter;
- list and tuple order must matter;
- set order must not matter;
- nested values must be handled recursively;
- unsupported mutable/custom objects must raise `TypeError` rather than silently using a fragile representation.


### Solution

Tag container types so that structurally different values do not collapse accidentally. Sorting mapping entries by a stable representation makes the canonical form deterministic within the supported domain.


In [12]:
_PRIMITIVES = (str, bytes, int, float, bool, type(None))


def deep_freeze(value: Any) -> Hashable:
    if isinstance(value, _PRIMITIVES):
        return value

    if isinstance(value, Mapping):
        frozen_items = [
            (deep_freeze(k), deep_freeze(v))
            for k, v in value.items()
        ]
        frozen_items.sort(key=repr)
        return ("mapping", tuple(frozen_items))

    if isinstance(value, list):
        return ("list", tuple(deep_freeze(item) for item in value))

    if isinstance(value, tuple):
        return ("tuple", tuple(deep_freeze(item) for item in value))

    if isinstance(value, (set, frozenset)):
        frozen_items = [deep_freeze(item) for item in value]
        frozen_items.sort(key=repr)
        return ("set", tuple(frozen_items))

    raise TypeError(f"Unsupported value for canonicalization: {type(value).__name__}")

config_a = {
    "layers": [64, 32],
    "options": {"dropout": 0.2, "activation": "relu"},
    "features": {"age", "income"},
}
config_b = {
    "features": {"income", "age"},
    "options": {"activation": "relu", "dropout": 0.2},
    "layers": [64, 32],
}

key_a = deep_freeze(config_a)
key_b = deep_freeze(config_b)
assert key_a == key_b
assert hash(key_a) == hash(key_b)
print("Canonical key hash:", hash(key_a))


Canonical key hash: -8926264971129441092


---
## Problem 8 — Build a robust function-call cache key

Design `make_call_key(function_name, args, kwargs)` so that:

- positional argument order matters;
- keyword argument order does not matter;
- nested supported containers are canonicalized;
- the function name is part of the key;
- calls to different functions cannot collide semantically even if arguments match.


### Solution


In [13]:
def make_call_key(
    function_name: str,
    args: tuple[Any, ...],
    kwargs: Mapping[str, Any],
) -> tuple[Hashable, ...]:
    return (
        "call",
        function_name,
        deep_freeze(args),
        deep_freeze(dict(kwargs)),
    )

key1 = make_call_key("train", ([1, 2],), {"epochs": 10, "verbose": False})
key2 = make_call_key("train", ([1, 2],), {"verbose": False, "epochs": 10})
key3 = make_call_key("evaluate", ([1, 2],), {"verbose": False, "epochs": 10})

assert key1 == key2
assert key1 != key3
cache = {key1: "trained-model"}
assert cache[key2] == "trained-model"
print("Cache key components:", len(key1))


Cache key components: 4


---
## Problem 9 — Instrument collisions and count equality checks

Two key classes store the same integer identity:

- `DistributedKey` hashes the integer;
- `ConstantHashKey` always returns `1`.

Measure both lookup time and the number of `__eq__` calls required for a successful lookup near the end of a dictionary.


### Solution

A collision does not make a dictionary incorrect. It makes Python inspect more candidate keys and perform more equality checks. A constant hash is valid but pathologically slow.


In [14]:
class EqualityCounter:
    comparisons = 0

    @classmethod
    def reset(cls) -> None:
        cls.comparisons = 0


class DistributedKey(EqualityCounter):
    __slots__ = ("value",)

    def __init__(self, value: int):
        self.value = value

    def __eq__(self, other: object) -> bool:
        type(self).comparisons += 1
        if not isinstance(other, DistributedKey):
            return NotImplemented
        return self.value == other.value

    def __hash__(self) -> int:
        return hash(self.value)


class ConstantHashKey(EqualityCounter):
    __slots__ = ("value",)

    def __init__(self, value: int):
        self.value = value

    def __eq__(self, other: object) -> bool:
        type(self).comparisons += 1
        if not isinstance(other, ConstantHashKey):
            return NotImplemented
        return self.value == other.value

    def __hash__(self) -> int:
        return 1

N = 2_000
distributed = {DistributedKey(i): i for i in range(N)}
constant = {ConstantHashKey(i): i for i in range(N)}

DistributedKey.reset()
assert distributed[DistributedKey(N - 1)] == N - 1
distributed_comparisons = DistributedKey.comparisons

ConstantHashKey.reset()
assert constant[ConstantHashKey(N - 1)] == N - 1
constant_comparisons = ConstantHashKey.comparisons

print("Equality checks with distributed hashes:", distributed_comparisons)
print("Equality checks with constant hash:", constant_comparisons)


Equality checks with distributed hashes: 1
Equality checks with constant hash: 2000


In [15]:
distributed_time = timeit(
    "distributed[DistributedKey(N - 1)]",
    globals=globals(),
    number=2_000,
)
constant_time = timeit(
    "constant[ConstantHashKey(N - 1)]",
    globals=globals(),
    number=2_000,
)

print(f"Distributed-hash lookup: {distributed_time:.6f}s")
print(f"Constant-hash lookup:    {constant_time:.6f}s")
print(f"Slowdown factor:         {constant_time / distributed_time:.1f}x")


Distributed-hash lookup: 0.006932s
Constant-hash lookup:    3.131161s
Slowdown factor:         451.7x


---
## Problem 10 — Cache an expensive hash for an immutable object

A `DocumentKey` contains a large immutable byte payload. Recomputing its digest-based hash on every dictionary access is wasteful.

Design a class that:

- is immutable;
- computes an integer hash from a cryptographic digest;
- caches the result after the first computation;
- keeps equality based on the original bytes, not merely on the digest.

Why must equality still compare the actual identity data?


### Solution

Digest collisions are extraordinarily unlikely but still possible. Hash equality only identifies candidates; equality must define semantic identity. The cached result is safe only because the object is immutable.


In [16]:
@dataclass(frozen=True, slots=True)
class DocumentKey:
    payload: bytes
    _cached_hash: int | None = field(default=None, init=False, repr=False, compare=False)

    def __hash__(self) -> int:
        cached = self._cached_hash
        if cached is None:
            digest = hashlib.blake2b(self.payload, digest_size=16).digest()
            cached = int.from_bytes(digest, "big", signed=False)
            object.__setattr__(self, "_cached_hash", cached)
        return cached

payload = b"abc123" * 100_000
doc = DocumentKey(payload)

first = timeit("hash(doc)", globals=globals(), number=1)
repeated = timeit("hash(doc)", globals=globals(), number=20_000)

assert doc == DocumentKey(payload)
assert_hash_contract(doc, DocumentKey(payload))
print(f"First hash call: {first:.6f}s")
print(f"20,000 cached calls: {repeated:.6f}s")


First hash call: 0.001936s
20,000 cached calls: 0.009949s


---
## Problem 11 — Distinguish `hash()` from a stable external identifier

A developer stores `hash(customer_key)` in a database and expects the value to remain identical across Python processes and deployments.

### Tasks

1. Demonstrate that string hashes may differ when subprocesses use different hash seeds.
2. Build a stable hexadecimal identifier from canonical JSON and BLAKE2.
3. Explain when to use each mechanism.


### Solution

Python's `hash()` is for in-process hash tables. It is not a serialization format, database identifier, signature, or cross-process partition key. Use an explicitly specified stable digest for those use cases.


In [17]:
def hash_in_subprocess(seed: str, value: str) -> int:
    env = os.environ.copy()
    env["PYTHONHASHSEED"] = seed
    output = subprocess.check_output(
        [sys.executable, "-c", f"print(hash({value!r}))"],
        env=env,
        text=True,
    )
    return int(output.strip())

sample = "customer:42"
h1 = hash_in_subprocess("1", sample)
h2 = hash_in_subprocess("2", sample)
print("Seed 1:", h1)
print("Seed 2:", h2)
print("Different:", h1 != h2)


Seed 1: -681652150076155593
Seed 2: 2607877256469055809
Different: True


In [18]:
def stable_json_id(value: Any, *, digest_size: int = 16) -> str:
    canonical = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.blake2b(canonical, digest_size=digest_size).hexdigest()

customer_a = {"region": "eu", "customer_id": 42}
customer_b = {"customer_id": 42, "region": "eu"}

assert stable_json_id(customer_a) == stable_json_id(customer_b)
print(stable_json_id(customer_a))


d83079a633804135fad33a4afbf4475e


---
## Problem 12 — Handle `bool`, `int`, and `float` key equivalence

Python intentionally treats several numeric values as equal:

```python
True == 1 == 1.0
```

Their hashes also agree, so they occupy one dictionary key slot.

### Tasks

1. Verify the behavior.
2. Design a `TypedNumber` key that distinguishes both value and exact type.


### Solution


In [19]:
numeric_dict = {True: "bool", 1: "int", 1.0: "float"}
print(numeric_dict)
print("length:", len(numeric_dict))
print("hashes:", hash(True), hash(1), hash(1.0))


{True: 'float'}
length: 1
hashes: 1 1 1


In [20]:
@dataclass(frozen=True, slots=True)
class TypedNumber:
    value: bool | int | float

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, TypedNumber):
            return NotImplemented
        return type(self.value) is type(other.value) and self.value == other.value

    def __hash__(self) -> int:
        return hash((type(self.value), self.value))

typed = {
    TypedNumber(True): "bool",
    TypedNumber(1): "int",
    TypedNumber(1.0): "float",
}
assert len(typed) == 3
print(typed)


{TypedNumber(value=True): 'bool', TypedNumber(value=1): 'int', TypedNumber(value=1.0): 'float'}


---
## Problem 13 — Explore `NaN` as a dictionary key

IEEE floating-point NaN is not equal to itself. Investigate the consequences for dictionary lookup and design a `CanonicalFloat` wrapper where all NaN values are considered equal.

Requirements:

- all NaN instances compare equal to one another;
- equal NaN wrappers have equal hashes;
- ordinary floats preserve normal equality semantics;
- `-0.0` and `0.0` remain equal, as in Python.


In [21]:
nan_a = float("nan")
nan_b = float("nan")
print("nan_a == nan_a:", nan_a == nan_a)
print("nan_a == nan_b:", nan_a == nan_b)

d = {nan_a: "stored"}
print("lookup with same object:", d[nan_a])
print("lookup with different NaN object:", d.get(nan_b, "not found"))


nan_a == nan_a: False
nan_a == nan_b: False
lookup with same object: stored
lookup with different NaN object: not found


### Solution


In [22]:
@dataclass(frozen=True, slots=True)
class CanonicalFloat:
    value: float

    @property
    def is_nan(self) -> bool:
        return math.isnan(self.value)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, CanonicalFloat):
            return NotImplemented
        if self.is_nan and other.is_nan:
            return True
        return self.value == other.value

    def __hash__(self) -> int:
        if self.is_nan:
            return hash((CanonicalFloat, "NaN"))
        return hash((CanonicalFloat, self.value))

cn1 = CanonicalFloat(float("nan"))
cn2 = CanonicalFloat(float("nan"))
cz1 = CanonicalFloat(-0.0)
cz2 = CanonicalFloat(0.0)

assert cn1 == cn2
assert hash(cn1) == hash(cn2)
assert cz1 == cz2
assert hash(cz1) == hash(cz2)
assert_mapping_round_trip(cn1, cn2)
print({cn1: "canonical NaN"}[cn2])


canonical NaN


---
## Problem 14 — Restore or suppress hashing in an inheritance hierarchy

Study three classes:

1. a parent with identity equality/hash;
2. a child that defines value equality but no hash;
3. a child that explicitly keeps identity semantics.

Predict and verify `__hash__` for each class.


### Solution

Defining `__eq__` in the child normally sets the child's `__hash__` to `None`. Assigning `__hash__ = Parent.__hash__` is only correct when the child's equality remains compatible with the parent's hash semantics. For pure identity semantics, inheriting both `object.__eq__` and `object.__hash__` is coherent.


In [23]:
class IdentityBase:
    pass


class ValueChild(IdentityBase):
    def __init__(self, value: int):
        self.value = value

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ValueChild):
            return NotImplemented
        return self.value == other.value


class ExplicitIdentityChild(IdentityBase):
    __eq__ = object.__eq__
    __hash__ = object.__hash__

print("IdentityBase.__hash__:", IdentityBase.__hash__)
print("ValueChild.__hash__:", ValueChild.__hash__)
print("ExplicitIdentityChild.__hash__:", ExplicitIdentityChild.__hash__)

try:
    hash(ValueChild(1))
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

x = ExplicitIdentityChild()
y = ExplicitIdentityChild()
assert x != y
assert len({x, y}) == 2


IdentityBase.__hash__: <slot wrapper '__hash__' of 'object' objects>
ValueChild.__hash__: None
ExplicitIdentityChild.__hash__: <slot wrapper '__hash__' of 'object' objects>
TypeError: unhashable type: 'ValueChild'


---
## Problem 15 — Deduplicate records while preserving first occurrence

Given a sequence of records, remove duplicates by `(tenant_id, external_id)` while preserving the first record for each identity. Other fields such as timestamps and payloads do not define identity.

Implement an immutable key object and an `O(n)` deduplication function.


### Solution


In [24]:
@dataclass(frozen=True, slots=True)
class RecordKey:
    tenant_id: str
    external_id: str


def deduplicate_records(records: Iterable[Mapping[str, Any]]) -> list[Mapping[str, Any]]:
    seen: set[RecordKey] = set()
    result: list[Mapping[str, Any]] = []

    for record in records:
        key = RecordKey(
            tenant_id=str(record["tenant_id"]),
            external_id=str(record["external_id"]),
        )
        if key not in seen:
            seen.add(key)
            result.append(record)

    return result

records = [
    {"tenant_id": "A", "external_id": "7", "payload": "first"},
    {"tenant_id": "A", "external_id": "7", "payload": "duplicate"},
    {"tenant_id": "B", "external_id": "7", "payload": "different tenant"},
]

unique = deduplicate_records(records)
assert [r["payload"] for r in unique] == ["first", "different tenant"]
print(unique)


[{'tenant_id': 'A', 'external_id': '7', 'payload': 'first'}, {'tenant_id': 'B', 'external_id': '7', 'payload': 'different tenant'}]


---
## Problem 16 — Create a versioned entity key with explicit identity evolution

An API evolves from identity `(namespace, name)` in version 1 to `(namespace, name, revision)` in version 2. Mixing versions silently would be dangerous.

Design keys so that:

- V1 and V2 keys never compare equal;
- each version is internally hash-consistent;
- the version is visible in the representation;
- migration from V1 to V2 is explicit.


### Solution


In [25]:
@dataclass(frozen=True, slots=True)
class EntityKeyV1:
    namespace: str
    name: str

    def migrate(self, revision: int) -> "EntityKeyV2":
        return EntityKeyV2(self.namespace, self.name, revision)


@dataclass(frozen=True, slots=True)
class EntityKeyV2:
    namespace: str
    name: str
    revision: int

v1 = EntityKeyV1("catalog", "item-42")
v2 = v1.migrate(revision=3)

assert v1 != v2
assert len({v1, v2}) == 2
print(v1)
print(v2)


EntityKeyV1(namespace='catalog', name='item-42')
EntityKeyV2(namespace='catalog', name='item-42', revision=3)


---
## Problem 17 — Build a stable partition key

You need to assign customer records to one of 32 partitions in a way that is stable across Python processes and deployments.

Do **not** use `hash(customer_id) % 32`.

Implement a partition function using a named encoding and digest algorithm. Then verify that equivalent text produces the same result.


### Solution

A stable partition scheme must fully specify normalization, encoding, digest algorithm, byte order, and partition count. Changing any of these is a data migration.


In [26]:
def stable_partition(customer_id: str, partitions: int) -> int:
    if partitions <= 0:
        raise ValueError("partitions must be positive")
    canonical = customer_id.strip().casefold().encode("utf-8")
    digest = hashlib.blake2b(canonical, digest_size=8).digest()
    number = int.from_bytes(digest, byteorder="big", signed=False)
    return number % partitions

assert stable_partition(" Customer-42 ", 32) == stable_partition("customer-42", 32)
print("Partition:", stable_partition("customer-42", 32))


Partition: 10


---
## Problem 18 — Property-style randomized testing of the contract

Write a randomized test for a normalized `ProductCode` key. Equivalent spellings differ in case and surrounding whitespace. The test should generate many equivalent pairs and assert:

- equality symmetry;
- equal hashes;
- set deduplication;
- dictionary retrieval.


### Solution

This is not a replacement for a property-testing library, but it demonstrates the invariants such a library should generate and verify.


In [27]:
@dataclass(frozen=True, slots=True)
class ProductCode:
    canonical: str

    def __post_init__(self) -> None:
        normalized = self.canonical.strip().upper()
        if not normalized or any(ch.isspace() for ch in normalized):
            raise ValueError("Product code must be non-empty and contain no internal whitespace")
        object.__setattr__(self, "canonical", normalized)


def randomized_product_code_test(seed: int = 2026, cases: int = 1_000) -> None:
    rng = random.Random(seed)
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

    for _ in range(cases):
        raw = "".join(rng.choice(alphabet) for _ in range(rng.randint(3, 12)))
        variant_a = " " * rng.randint(0, 3) + raw.lower() + " " * rng.randint(0, 3)
        variant_b = " " * rng.randint(0, 3) + raw.upper() + " " * rng.randint(0, 3)

        a = ProductCode(variant_a)
        b = ProductCode(variant_b)

        assert a == b
        assert b == a
        assert hash(a) == hash(b)
        assert len({a, b}) == 1
        assert {a: "value"}[b] == "value"

randomized_product_code_test()
print("1,000 randomized contract cases passed")


1,000 randomized contract cases passed


---
## Problem 19 — Capstone: production-style query cache keys

Build a key for an analytics query cache. Query identity consists of:

- normalized SQL text;
- positional parameters;
- named parameters independent of order;
- data-source name;
- schema version.

Operational fields such as request ID, submission time, and caller name must not affect cache identity.

### Additional requirements

- The key must be immutable.
- Nested parameters must be canonicalized.
- SQL normalization should collapse runs of whitespace and trim leading/trailing whitespace.
- Two equivalent requests should retrieve the same cache entry.
- A schema-version change must produce a different key.


### Solution


In [28]:
def normalize_sql(sql: str) -> str:
    return " ".join(sql.split())


@dataclass(frozen=True, slots=True)
class QueryKey:
    source: str
    schema_version: int
    sql: str
    positional: Hashable
    named: Hashable

    @classmethod
    def build(
        cls,
        *,
        source: str,
        schema_version: int,
        sql: str,
        positional: Sequence[Any] = (),
        named: Mapping[str, Any] | None = None,
    ) -> "QueryKey":
        if schema_version < 1:
            raise ValueError("schema_version must be at least 1")
        return cls(
            source=source.strip().casefold(),
            schema_version=schema_version,
            sql=normalize_sql(sql),
            positional=deep_freeze(tuple(positional)),
            named=deep_freeze(dict(named or {})),
        )


@dataclass(slots=True)
class QueryRequest:
    key: QueryKey
    request_id: str
    caller: str

cache: dict[QueryKey, list[tuple[Any, ...]]] = {}

request_a = QueryRequest(
    key=QueryKey.build(
        source="Warehouse",
        schema_version=3,
        sql="SELECT  *\nFROM sales WHERE region = :region",
        named={"region": "EU", "filters": ["paid", "shipped"]},
    ),
    request_id="req-001",
    caller="dashboard",
)

request_b = QueryRequest(
    key=QueryKey.build(
        source=" warehouse ",
        schema_version=3,
        sql="  SELECT * FROM sales   WHERE region = :region  ",
        named={"filters": ["paid", "shipped"], "region": "EU"},
    ),
    request_id="req-999",
    caller="scheduled-report",
)

cache[request_a.key] = [(1001, "EU")]
assert cache[request_b.key] == [(1001, "EU")]
assert request_a.key == request_b.key

new_schema_key = QueryKey.build(
    source="warehouse",
    schema_version=4,
    sql="SELECT * FROM sales WHERE region = :region",
    named={"region": "EU", "filters": ["paid", "shipped"]},
)
assert new_schema_key != request_a.key

print("Cache hit:", cache[request_b.key])
print("Different schema version creates new key:", new_schema_key != request_a.key)


Cache hit: [(1001, 'EU')]
Different schema version creates new key: True


---
## Problem 20 — Review and repair an intentionally flawed key class

The following design has multiple defects:

```python
class SessionKey:
    def __init__(self, user, permissions=[]):
        self.user = user
        self.permissions = permissions

    def __eq__(self, other):
        return self.user.lower() == other.user.lower()

    def __hash__(self):
        return hash((self.user, tuple(self.permissions)))
```

Identify at least six issues, then implement a robust replacement.


### Solution discussion

Problems include:

1. A mutable default list is shared across instances.
2. `__eq__` assumes `other` has a `.user` attribute.
3. Equality normalizes case, but hashing does not.
4. Permissions participate in hashing but not equality, violating the contract.
5. Both `user` and `permissions` are mutable after insertion.
6. The intended meaning of permissions is ambiguous: identity or metadata?
7. Input normalization and validation are missing.
8. The class exposes a mutable list directly.

The repaired design below treats normalized user plus an unordered permission set as identity. If permissions are metadata instead, remove them from both equality and hashing and store them in a separate mutable record.


In [29]:
@dataclass(frozen=True, slots=True)
class SessionKey:
    user: str
    permissions: frozenset[str] = field(default_factory=frozenset)

    def __post_init__(self) -> None:
        normalized_user = self.user.strip().casefold()
        if not normalized_user:
            raise ValueError("user must not be empty")

        normalized_permissions = frozenset(
            permission.strip().casefold()
            for permission in self.permissions
            if permission.strip()
        )

        object.__setattr__(self, "user", normalized_user)
        object.__setattr__(self, "permissions", normalized_permissions)

s1 = SessionKey(" Ada ", frozenset({"READ", "write"}))
s2 = SessionKey("ada", frozenset({"write", "read"}))

assert s1 == s2
assert hash(s1) == hash(s2)
assert_mapping_round_trip(s1, s2)
print(s1)


SessionKey(user='ada', permissions=frozenset({'write', 'read'}))


---
# Additional advanced drills with concise solutions

These shorter problems reinforce design choices that frequently appear in code reviews.


## Drill A — Make a class deliberately unhashable

A mutable shopping cart compares by its current line items. It must not be used as a set element or dictionary key.

### Solution

Define value equality and explicitly set `__hash__ = None` to document the intent.


In [30]:
class ShoppingCart:
    __hash__ = None

    def __init__(self, items: Iterable[str] = ()):
        self.items = list(items)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ShoppingCart):
            return NotImplemented
        return self.items == other.items

cart = ShoppingCart(["book"])
try:
    hash(cart)
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: unhashable type: 'ShoppingCart'


## Drill B — Explain why `return hash(str(self))` is usually weak design

### Solution

`str(self)` is presentation, not necessarily identity. Formatting may change, omit fields, be locale-dependent, or include non-identity metadata. Hash the exact immutable equality tuple instead.


In [31]:
@dataclass(frozen=True, slots=True)
class Coordinate3D:
    x: int
    y: int
    z: int

    def __hash__(self) -> int:
        return hash((self.x, self.y, self.z))

assert_hash_contract(Coordinate3D(1, 2, 3), Coordinate3D(1, 2, 3))


## Drill C — Understand hash truncation

`__hash__` must return an integer, but `hash(obj)` may normalize a very large integer to the platform's hash width/modulus.


In [32]:
class HugeHash:
    def __hash__(self) -> int:
        return 10**100

obj = HugeHash()
print("Direct __hash__ result digits:", len(str(obj.__hash__())))
print("Normalized hash(obj):", hash(obj))
print("Platform width:", sys.hash_info.width)


Direct __hash__ result digits: 101
Normalized hash(obj): 910685213754167845
Platform width: 64


## Drill D — Validate a whole equivalence class

Three differently formatted phone keys should form one equivalence class.


In [33]:
@dataclass(frozen=True, slots=True)
class PhoneKey:
    digits: str

    def __post_init__(self) -> None:
        normalized = "".join(ch for ch in self.digits if ch.isdigit())
        if len(normalized) < 7:
            raise ValueError("phone number is too short")
        object.__setattr__(self, "digits", normalized)

p1 = PhoneKey("+1 (555) 010-2000")
p2 = PhoneKey("1-555-010-2000")
p3 = PhoneKey("15550102000")
assert_equivalence_laws(p1, p2, p3)
assert len({p1, p2, p3}) == 1
print(p1)


PhoneKey(digits='15550102000')


---
# Best-practices checklist

Before making a custom object hashable, verify all of the following:

1. **Define identity first.** Write down exactly which fields make two instances equal.
2. **Use the same identity fields in `__eq__` and `__hash__`.**
3. **Prefer immutability.** Frozen dataclasses are a strong default for value keys.
4. **Return `NotImplemented` for unsupported comparisons.**
5. **Choose subclass behavior deliberately.** Use strict type equality when subclasses may add identity.
6. **Never use a mutable field in the hash.** This includes lists, dictionaries, sets, and mutable nested objects.
7. **Do not require unequal objects to have unequal hashes.** Collisions are allowed, but excessive collisions hurt performance.
8. **Use tuple hashing for ordinary composite keys.** Example: `hash((field_a, field_b))`.
9. **Separate immutable identity from mutable metadata.**
10. **Do not persist Python's `hash()` output.** Use a specified stable digest for cross-process IDs or partitions.
11. **Test dictionary/set behavior, not only direct equality.**
12. **Test normalization boundaries and unusual values.** Examples: Unicode case folding, NaN, signed zero, booleans versus integers, and version changes.


# Final challenge prompts

Use the patterns in this notebook to implement and test these without looking back at the solutions:

1. An immutable `IPv4NetworkKey` that normalizes host bits to the network address.
2. A `FileFingerprint` key based on file size plus BLAKE2 digest, with cached hashing.
3. A case-sensitive repository path key but case-insensitive Windows drive-letter normalization.
4. A geographic tile key `(zoom, x, y)` with strict range validation.
5. A graph edge key where `(A, B)` equals `(B, A)` for undirected graphs.
6. A directed edge key where order matters.
7. A semantic-version key that ignores build metadata for precedence but not for exact artifact identity—requiring two distinct key types.
8. A database composite-primary-key class generated from a schema definition.
9. A request-id wrapper that rejects mutable payload data from participating in identity.
10. A benchmark comparing tuple keys, frozen dataclass keys, and constant-hash keys.
